# Deep Dive — Transfer Learning: Fine-Tuning a Pretrained Model

**Module 4: CNNs, Pretrained Models & Deep Dream | Optional Deep Dive**

---

In the lectures, we used pretrained models for classification and feature visualization. But what if you want to classify something that's *not* in ImageNet's 1000 classes?

**Transfer learning:** Take a model pretrained on ImageNet, replace the final classification layer with your own, and train it on your dataset. The pretrained features (edges, textures, shapes) transfer to your new task.

This is a preview of a technique you'll use heavily in Module 6 (fine-tuning transformers).

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
import torchvision.transforms as transforms
from torchvision import datasets
from torch.utils.data import DataLoader, Subset
import matplotlib.pyplot as plt
import numpy as np

---

## The Task: CIFAR-10 with a Pretrained ResNet

We'll compare three approaches:
1. **Train from scratch** — random initialization
2. **Feature extraction** — freeze pretrained layers, only train the new head
3. **Fine-tuning** — unfreeze the last few layers and train them too

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# CIFAR-10 with ImageNet-style preprocessing
transform_train = transforms.Compose([
    transforms.Resize(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

transform_test = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# Use a subset for speed (5000 train, 1000 test)
full_train = datasets.CIFAR10('data', train=True, download=True, transform=transform_train)
full_test = datasets.CIFAR10('data', train=False, transform=transform_test)

torch.manual_seed(42)
train_idx = torch.randperm(len(full_train))[:5000]
test_idx = torch.randperm(len(full_test))[:1000]

train_data = Subset(full_train, train_idx)
test_data = Subset(full_test, test_idx)

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
test_loader = DataLoader(test_data, batch_size=64)

print(f"Training: {len(train_data)}, Test: {len(test_data)}")
print(f"Device: {device}")

In [ ]:
def train_and_evaluate(model, train_loader, test_loader, device, epochs=5, lr=0.001):
    model = model.to(device)
    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    accs = []

    for epoch in range(epochs):
        model.train()
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            loss_fn(model(images), labels).backward()
            optimizer.step()

        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for images, labels in test_loader:
                images, labels = images.to(device), labels.to(device)
                correct += (model(images).argmax(1) == labels).sum().item()
                total += labels.size(0)
        accs.append(correct / total)
        print(f"  Epoch {epoch+1}: {accs[-1]:.2%}")

    return accs

In [ ]:
# Approach 1: From scratch
print("=== From Scratch ===")
torch.manual_seed(42)
scratch = models.resnet18(weights=None)  # no pretrained weights
scratch.fc = nn.Linear(512, 10)
trainable = sum(p.numel() for p in scratch.parameters() if p.requires_grad)
print(f"Trainable params: {trainable:,}")
acc_scratch = train_and_evaluate(scratch, train_loader, test_loader, device)

In [ ]:
# Approach 2: Feature extraction (freeze everything except the new head)
print("=== Feature Extraction ===")
feature_ext = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
for param in feature_ext.parameters():
    param.requires_grad = False  # freeze all
feature_ext.fc = nn.Linear(512, 10)  # new head (trainable)
trainable = sum(p.numel() for p in feature_ext.parameters() if p.requires_grad)
print(f"Trainable params: {trainable:,}")
acc_feature = train_and_evaluate(feature_ext, train_loader, test_loader, device)

In [ ]:
# Approach 3: Fine-tuning (unfreeze last few layers)
print("=== Fine-Tuning ===")
finetune = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
for param in finetune.parameters():
    param.requires_grad = False
# Unfreeze layer4 and fc
for param in finetune.layer4.parameters():
    param.requires_grad = True
finetune.fc = nn.Linear(512, 10)
trainable = sum(p.numel() for p in finetune.parameters() if p.requires_grad)
print(f"Trainable params: {trainable:,}")
acc_finetune = train_and_evaluate(finetune, train_loader, test_loader, device, lr=0.0001)

In [ ]:
# Compare
fig, ax = plt.subplots(figsize=(10, 5))
epochs = range(1, 6)
ax.plot(epochs, [a*100 for a in acc_scratch], 'o-', label=f'From scratch ({acc_scratch[-1]:.1%})',
       color='#D9534F', linewidth=2)
ax.plot(epochs, [a*100 for a in acc_feature], 's-', label=f'Feature extraction ({acc_feature[-1]:.1%})',
       color='#4A90D9', linewidth=2)
ax.plot(epochs, [a*100 for a in acc_finetune], '^-', label=f'Fine-tuning ({acc_finetune[-1]:.1%})',
       color='#5CB85C', linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('Test Accuracy (%)')
ax.set_title('Transfer Learning Comparison', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**Feature extraction** gets decent accuracy almost immediately — the pretrained features are already useful. **Fine-tuning** does even better because it adapts the last few layers to CIFAR-10's specific patterns. **From scratch** is worst because 5000 images isn't enough to learn good features from nothing.

The less data you have, the more transfer learning helps.

---

## Exercises

1. Try unfreezing `layer3` in addition to `layer4`. Does it improve accuracy?
2. What happens if you use only 500 training images instead of 5000? Which approach degrades the most?
3. Try using MobileNetV2 instead of ResNet-18. Does the smaller model transfer as well?

In [ ]:
# Exercises: Your code here
